# 02 — Data Pipeline

Tokenize & preprocess text data into efficient binary shards for training.

**Output:** Memory-mapped `.bin` files (fast, CPU-efficient, works with BitNet).

> Runs on **Google Colab** (free T4 tier).
> Format inspired by NanoGPT — each shard is a flat `uint16` array of token IDs.

## 1. Install dependencies

In [ ]:
!pip install -q tokenizers datasets tqdm numpy

## 2. Mount Google Drive (optional)

For saving shards persistently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

In [ ]:
import os, math, time, random
import numpy as np
from tqdm.auto import tqdm
from tokenizers import Tokenizer
from datasets import load_dataset

# ─── Tokenizer ───
TOKENIZER_PATH = "anthor_tokenizer.json"  # from 01_tokenizer.ipynb

# ─── Data ───
DATASET_NAME   = "HuggingFaceFW/fineweb-edu"
DATASET_CONFIG = "sample-10BT"
DATASET_SPLIT  = "train"
MAX_DOCS       = 20_000      # ~2GB raw text — adjust for more/less

# ─── Sharding ───
TOKENS_PER_SHARD = 50_000_000  # 50M tokens per shard (~100MB each)
VAL_SPLIT        = 0.001       # 0.1% held out for validation

# ─── Paths ───
SHARD_DIR     = "data_shards"
VAL_SHARD_DIR = "data_shards_val"
os.makedirs(SHARD_DIR, exist_ok=True)
os.makedirs(VAL_SHARD_DIR, exist_ok=True)

## 4. Load tokenizer

In [ ]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
vocab_size = tokenizer.get_vocab_size()
print(f"Loaded tokenizer: vocab_size={vocab_size}")
print(f"Special tokens: {tokenizer.token_to_id('<pad>')}, <unk>={tokenizer.token_to_id('<unk>')}, "
      f"<s>={tokenizer.token_to_id('<s>')}, </s>={tokenizer.token_to_id('</s>')}")

## 5. Helper: tokenize & write shard

Encodes documents sequentially, flushes to disk every `TOKENS_PER_SHARD` tokens.

In [ ]:
def write_shard(tokens, shard_dir, shard_idx):
    """Save token array as uint16 .bin shard."""
    arr = np.array(tokens, dtype=np.uint16)
    path = os.path.join(shard_dir, f"shard_{shard_idx:05d}.bin")
    arr.tofile(path)
    return path, len(arr)


def build_dataset(docs, tokenizer, tokens_per_shard, shard_dir, desc="Processing"):
    """Tokenize docs, write shards, return stats."""
    buffer = []
    shard_idx = 0
    total_tokens = 0
    total_chars = 0
    shard_sizes = []

    pbar = tqdm(docs, desc=desc)
    for doc in pbar:
        enc = tokenizer.encode(doc)
        ids = enc.ids
        total_chars += len(doc)
        buffer.extend(ids)

        while len(buffer) >= tokens_per_shard:
            chunk = buffer[:tokens_per_shard]
            buffer = buffer[tokens_per_shard:]
            path, size = write_shard(chunk, shard_dir, shard_idx)
            total_tokens += size
            shard_sizes.append(size)
            shard_idx += 1
            pbar.set_postfix({"shards": shard_idx, "tokens": f"{total_tokens/1e6:.1f}M"})

    # Flush remaining
    if buffer:
        path, size = write_shard(buffer, shard_dir, shard_idx)
        total_tokens += size
        shard_sizes.append(size)
        shard_idx += 1

    return {
        "shards": shard_idx,
        "total_tokens": total_tokens,
        "total_chars": total_chars,
        "shard_sizes": shard_sizes,
        "chars_per_token": total_chars / max(total_tokens, 1),
    }

## 6. Load & tokenize data

Streaming from FineWeb-Edu, tokenize on-the-fly, write shards.

> **Warning:** `MAX_DOCS=20,000` → ~100-200M tokens → ~7-15 shards.
> On Colab this takes **5-15 minutes** depending on network & CPU.
> Reduce `MAX_DOCS` if you just want to test the pipeline.

In [ ]:
print(f"Loading dataset: {DATASET_NAME} ({DATASET_CONFIG}/{DATASET_SPLIT})")
ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT, streaming=True)

# Collect docs (limit to MAX_DOCS)
docs = []
for i, example in enumerate(ds):
    if i >= MAX_DOCS:
        break
    docs.append(example["text"])
    if (i + 1) % 5000 == 0:
        print(f"  Collected {i + 1}/{MAX_DOCS}")

total_bytes = sum(len(d.encode("utf-8")) for d in docs)
print(f"\nLoaded {len(docs)} docs, ~{total_bytes / 1e6:.1f} MB raw text")

## 7. Train / Validation split

Randomly split docs (not tokens) to avoid cross-contamination.

In [ ]:
random.seed(42)
random.shuffle(docs)

n_val = max(1, int(len(docs) * VAL_SPLIT))
val_docs = docs[:n_val]
train_docs = docs[n_val:]

print(f"Train docs: {len(train_docs)}")
print(f"Val docs:   {len(val_docs)}")

## 8. Process training shards

In [ ]:
t0 = time.time()
train_stats = build_dataset(
    train_docs, tokenizer, TOKENS_PER_SHARD, SHARD_DIR, desc="Training"
)
elapsed = time.time() - t0

print(f"\n{'='*50}")
print(f"Train set done in {elapsed:.0f}s")
print(f"  Shards:       {train_stats['shards']}")
print(f"  Total tokens: {train_stats['total_tokens'] / 1e6:.2f}M")
print(f"  Total chars:  {train_stats['total_chars'] / 1e6:.2f}M")
print(f"  Chars/token:  {train_stats['chars_per_token']:.2f}")
print(f"  Avg shard:    {np.mean(train_stats['shard_sizes']) / 1e6:.2f}M tokens")
print(f"{'='*50}")

# List first few shards
shards = sorted(os.listdir(SHARD_DIR))
print(f"\nShard files ({len(shards)}):")
for s in shards[:5]:
    size_mb = os.path.getsize(os.path.join(SHARD_DIR, s)) / 1e6
    print(f"  {s}  {size_mb:.1f} MB")
if len(shards) > 5:
    print(f"  ... and {len(shards) - 5} more")

## 9. Process validation shards

In [ ]:
t0 = time.time()
val_stats = build_dataset(
    val_docs, tokenizer, min(TOKENS_PER_SHARD, 1_000_000), VAL_SHARD_DIR, desc="Validation"
)
elapsed = time.time() - t0

print(f"\nVal set done in {elapsed:.0f}s")
print(f"  Shards:       {val_stats['shards']}")
print(f"  Total tokens: {val_stats['total_tokens'] / 1e6:.2f}M")
print(f"  Chars/token:  {val_stats['chars_per_token']:.2f}")
print(f"{'='*50}")

## 10. Inspect shard content

Read back a shard to verify integrity.

In [ ]:
def inspect_shard(shard_path, tokenizer, n_tokens=50):
    """Load a shard and show first n_tokens decoded."""
    arr = np.fromfile(shard_path, dtype=np.uint16)
    print(f"Shard: {os.path.basename(shard_path)}")
    print(f"  Total tokens: {len(arr):,}")
    print(f"  Size: {os.path.getsize(shard_path) / 1e6:.1f} MB")
    print(f"  dtype: {arr.dtype}, min={arr.min()}, max={arr.max()}")

    # Decode first n_tokens
    prefix = arr[:n_tokens].tolist()
    text = tokenizer.decode(prefix)
    print(f"\n  First {n_tokens} tokens decoded:")
    print(f"  {repr(text[:200])}")
    print(f"\n  First 20 token IDs: {prefix[:20]}")
    return arr


# Inspect first train shard
train_shards = sorted(os.listdir(SHARD_DIR))
if train_shards:
    print("─" * 50)
    inspect_shard(os.path.join(SHARD_DIR, train_shards[0]), tokenizer)
    print("─" * 50)

# Inspect validation shard if exists
val_shards = sorted(os.listdir(VAL_SHARD_DIR))
if val_shards:
    inspect_shard(os.path.join(VAL_SHARD_DIR, val_shards[0]), tokenizer)

## 11. Save dataset metadata

JSON file with stats — useful for the training script to know vocab size, shard count, tokens per shard.

In [ ]:
import json

meta = {
    "vocab_size": vocab_size,
    "train_shards": train_stats["shards"],
    "val_shards": val_stats["shards"],
    "total_train_tokens": train_stats["total_tokens"],
    "total_val_tokens": val_stats["total_tokens"],
    "tokens_per_shard": TOKENS_PER_SHARD,
    "max_docs": MAX_DOCS,
    "chars_per_token": train_stats["chars_per_token"],
    "shard_dir": SHARD_DIR,
    "val_shard_dir": VAL_SHARD_DIR,
}

with open("dataset_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("dataset_meta.json written:")
print(json.dumps(meta, indent=2))

## 12. (Optional) Copy to Google Drive

In [ ]:
import shutil

drive_mounted = os.path.exists("/content/drive/MyDrive/")

if drive_mounted:
    dst = "/content/drive/MyDrive/ANTHOR/data_shards"
    shutil.copytree(SHARD_DIR, dst, dirs_exist_ok=True)
    shutil.copytree(VAL_SHARD_DIR, dst + "_val", dirs_exist_ok=True)
    shutil.copy("dataset_meta.json", "/content/drive/MyDrive/ANTHOR/")
    print(f"Copied to Drive: {dst}")
else:
    print("Drive not mounted — skipping Drive backup.")

---

## ✅ Next steps

1. Data is tokenized & sharded → proceed to **`03_model.ipynb`**
2. Training loop can memory-map shards directly (no RAM duplication)
3. Metadata at `dataset_meta.json` tells the trainer where shards are

| File | Content |
|------|---------|
| `data_shards/` | `shard_00000.bin` … `shard_000XX.bin` (train) |
| `data_shards_val/` | `shard_00000.bin` … (validation) |
| `dataset_meta.json` | Vocab size, shard count, tokens/shard |

> **Tip:** Delete `data_shards/` and `data_shards_val/` from Colab if you copy to Drive — they can be large.